# 📓 Notebook 01 — Web Application Offensive

SQLi, XSS, SSRF, IDOR, JWT, auth bypass, LFI/RFI, deserialization, API abuse.
Targets: Juice Shop, DVWA, WebGoat, bWAPP, crAPI, VAmPI.

# 🧱 BASE (shared setup) — run these first, in order

In [ ]:
# Which GPU did you actually get? Decide the model from this.
!nvidia-smi
import torch, psutil
p = torch.cuda.get_device_properties(0)
print(f"GPU: {p.name} | VRAM {p.total_memory/1e9:.1f} GB | RAM {psutil.virtual_memory().total/1e9:.1f} GB")
!df -h /content | tail -1

Wed Oct 14 07:26:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8             13W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

GPU: Tesla T4 | VRAM 15.6 GB | RAM 13.6 GB
overlay         236G   66G  171G  28% /


In [ ]:
import os
try:
    if not os.path.exists('/content/drive/MyDrive'):
        from google.colab import drive
        drive.mount('/content/drive')
    CACHE = '/content/drive/MyDrive/redteam'
    os.makedirs(CACHE, exist_ok=True)
    print("cache:", CACHE)
except Exception as e:
    print("Drive notice:", e)
    CACHE = '/content/redteam'
    os.makedirs(CACHE, exist_ok=True)
    print("local cache:", CACHE)


cache: /content/drive/MyDrive/redteam


In [ ]:
%%bash
apt-get -qq update
apt-get -qq install -y docker.io git curl wget jq unzip tmux nodejs npm python3-venv python3-pip net-tools iputils-ping dnsutils
# Colab blocks Docker's iptables hooks -> disable them
dockerd --iptables=false --ip6tables=false --bridge=none >/var/log/dockerd.log 2>&1 &
for i in $(seq 1 30); do docker info >/dev/null 2>&1 && break; sleep 2; done
docker info | grep -E "Server Version|Storage Driver"
echo "docker ready" 

Server Version: 29.1.3-colab
Storage Driver: vfs
docker ready


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


In [ ]:
%%bash
curl -fsSL https://ollama.com/install.sh | sh || true
command -v ollama || { curl -L https://ollama.com/download/ollama-linux-amd64.tgz | tar -xz -C /usr; }
nohup ollama serve >/var/log/ollama.log 2>&1 &
for i in $(seq 1 40); do curl -s localhost:11434/api/version >/dev/null && break; sleep 2; done
curl -s localhost:11434/api/version && echo " <- ollama up" 

/usr/local/bin/ollama
{"version":"0.40.4"} <- ollama up


>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
%%bash
if ollama list | grep -q "qwen2.5-coder:7b"; then
  echo "qwen2.5-coder:7b ready"
else
  ollama pull qwen2.5-coder:7b
fi
ollama list


qwen2.5-coder:7b ready
NAME                ID              SIZE      MODIFIED       
gpt-oss:20b         f38aa0c53da5    13 GB     6 minutes ago     
qwen2.5-coder:7b    dae161e27b0e    4.7 GB    27 minutes ago    


In [ ]:
%%bash
# HexStrike = 150+ tools behind one MCP endpoint
if [ ! -d /content/hexstrike ]; then
  git clone --depth 1 https://github.com/0x4m4/hexstrike-ai /content/hexstrike
fi
cd /content/hexstrike
if ! curl -sf localhost:8888/health >/dev/null; then
  nohup python3 hexstrike_server.py --port 8888 >/var/log/hexstrike.log 2>&1 &
  sleep 5
fi
echo "hexstrike:"; curl -s localhost:8888/health; echo

# Kali toolbox container the agent can shell into
docker pull -q kalilinux/kali-rolling || true
docker run -d --name redteam --network host kalilinux/kali-rolling sleep infinity || true
echo toolbox-ready


hexstrike:
{"all_essential_tools_available":false,"cache_stats":{"evictions":0,"hit_rate":"7.3%","hits":74,"max_size":1000,"misses":942,"size":12},"category_stats":{"additional":{"available":0,"total":14},"api":{"available":1,"total":8},"binary":{"available":3,"total":13},"cloud":{"available":0,"total":10},"essential":{"available":0,"total":8},"exploitation":{"available":0,"total":3},"forensics":{"available":2,"total":16},"network":{"available":0,"total":10},"osint":{"available":1,"total":13},"password":{"available":0,"total":5},"vuln_scanning":{"available":1,"total":4},"web_security":{"available":4,"total":19},"wireless":{"available":0,"total":4}},"message":"HexStrike AI Tools API Server is operational","status":"healthy","telemetry":{"average_execution_time":"0.00s","commands_executed":942,"success_rate":"1.3%","system_metrics":{"cpu_percent":3.0,"disk_usage":27.7,"memory_percent":17.3,"network_io":{"bytes_recv":25361068180,"bytes_sent":68928118,"dropin":0,"dropout":0,"errin":0,"erro

In [ ]:
%%bash
if ! pgrep -f "cloudflared tunnel --url http://localhost:8888" >/dev/null; then
  wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared 2>/dev/null || true
  chmod +x /usr/local/bin/cloudflared 2>/dev/null || true
  nohup cloudflared tunnel --url http://localhost:8888 >/var/log/cf.log 2>&1 &
  sleep 10
fi
echo "HexStrike URL:"; grep -o 'https://[a-z0-9-]*\.trycloudflare\.com' /var/log/cf.log | head -1


HexStrike URL:
https://modification-similarly-retention-theatre.trycloudflare.com


In [ ]:
%%bash
# --- install Go for ProjectDiscovery tooling ---
if ! command -v go >/dev/null; then
  curl -fsSL https://go.dev/dl/go1.23.5.linux-amd64.tar.gz -o /tmp/go.tgz
  rm -rf /usr/local/go && tar -C /usr/local -xzf /tmp/go.tgz
  echo 'export PATH=$PATH:/usr/local/go/bin' >> /etc/profile.d/go.sh
fi
export PATH=$PATH:/usr/local/go/bin
go version

go version go1.23.5 linux/amd64


In [ ]:
%%bash
export PATH=/data/data/com.termux/files/home/.gemini/antigravity-cli/bin:/data/data/com.termux/files/home/.grok/bin:/data/data/com.termux/files/home/.local/bin:/data/data/com.termux/files/home/.local/bin:/data/data/com.termux/files/home/.local/bin:/data/data/com.termux/files/usr/bin:/usr/local/go/bin:/root/go/bin
for tool in nuclei httpx subfinder katana ffuf; do
  if ! command -v $tool >/dev/null; then
    echo "Installing $tool..."
    go install -v github.com/projectdiscovery/$tool/v3/cmd/$tool@latest 2>/dev/null || true
  fi
done
nuclei -version
nuclei -update-templates -silent || true
echo "ProjectDiscovery tools ready"


ProjectDiscovery tools ready


[INF] Nuclei Engine Version: v3.3.8
[INF] Nuclei Config Directory: /root/.config/nuclei
[INF] Nuclei Cache Directory: /root/.cache/nuclei
[INF] PDCP Directory: /root/.pdcp


In [ ]:
%%bash
apt-get -qq install -y sqlmap nikto whatweb wafw00f gobuster feroxbuster
pip -q install arjun commix 2>/dev/null || true
git clone --depth 1 https://github.com/ticarpi/jwt_tool /content/jwt_tool
git clone --depth 1 https://github.com/s0md3v/Arjun /content/arjun 2>/dev/null || true
docker pull -q zaproxy/zap-stable
docker run -d --name zap --network host zaproxy/zap-stable zap.sh -daemon -host 0.0.0.0 -port 8090 -config api.disablekey=true
echo "tools ready" 

Info: downloading layer sha256:460db26ea98b46e2bf2561d9aa120d9cc2522f1ad2c2b8244d118cbfd8b1dff3
Info: downloading layer sha256:b4d9080a546502e115c00b28dde75c295b34d4500a6f8ed5d082673081b7f14e
Info: downloading layer sha256:f034b4200fd633b11e562c5edfbf9224d8376116b6465bf3dfe4b03639aef8cd
Info: downloading layer sha256:9c316a78fb1d2ba53715557d6fe2fb6941809001e5611198bea0c7cb3c4809b5
Info: downloading layer sha256:630c1c3b0b458cbbfe50c6d6a791bec5cef28a2babce134a9674e566cb1da733
Info: downloading layer sha256:26594374861cbcad47b6c98fd8c72bff1bdc4121825b01e01f776f2f27b0d55b
Info: downloading layer sha256:a906b224ce5dc4679cc56a43861ea0cb7e32a0d7d22f20fb2998540f7314f74e
Info: downloading layer sha256:4c2883a5f689084d2cb4962ed3c1906d41ec63431b04cce9a6dd3889dbac10c3
Info: downloading layer sha256:4d50ae0cc8af006cf3a168656370a68eaa9a7c876beb2a9e008f525c31894cdf
Info: downloading layer sha256:90731ab28e80226cf8c4f95c08a65d851244d36ce01a41031816c8a1c745bb34
Info: downloading layer sha256:47737c69d

E: Unable to locate package feroxbuster
fatal: destination path '/content/jwt_tool' already exists and is not an empty directory.


In [ ]:
%%bash
# --- spin the labs (all --network host, distinct ports) ---
docker run -d --name juice   --network host bkimminich/juice-shop
docker run -d --name dvwa    --network host vulnerables/web-dvwa
docker run -d --name webgoat --network host webgoat/webgoat
docker run -d --name bwap    --network host raesene/bwapp
docker run -d --name vampi   --network host erev0s/vampi
sleep 30
for p in 3000 80 8080; do curl -s -o /dev/null -w "port $p -> %{http_code}\n" http://localhost:$p; done

Container juice started
Container dvwa started
Container webgoat started
Container bwap started
Container vampi started
port 3000 -> 200
port 80 -> 302
port 8080 -> 404


In [ ]:
# --- one-shot recon sweep the agent can riff on ---
target = "http://localhost:3000"   # Juice Shop
!httpx -u {target} -title -tech-detect -status-code -silent
!whatweb {target}
!wafw00f {target}
!nuclei -u {target} -severity low,medium,high,critical -silent | head -40

Usage: httpx [OPTIONS] URL

Error: No such option '-u'.


/bin/bash: line 1: whatweb: command not found


/bin/bash: line 1: wafw00f: command not found
